# Primer ensayo NOX en Colab gratuito

Cuaderno preparado el 26/10/2026; **todavía no ejecutado en Colab**. Objetivo: comprobar GPU y herramientas, verificar los datos y ejecutar un ensayo de 3 épocas con YOLO11n. No demuestra calidad comercial.

En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** (por ejemplo T4 si se ofrece). Usar la modalidad gratuita; no comprar unidades. Ejecutar las celdas una a una. Si no se asigna GPU, parar y registrar el mensaje.

Datos: 15 imágenes de entrenamiento y 3 de validación, con un solo positivo de validación en muñequera. No hay test independiente. Los grupos Milano/Germany son provisionales. Antes del bloque de subida y entrenamiento, registrar permisos del material y revisar la licencia de biblioteca/modelo; elegir Colab no confirma esos permisos.

Referencias: [Colab y sus límites](https://research.google.com/colaboratory/faq.html), [entrenamiento Ultralytics](https://docs.ultralytics.com/modes/train/), [YOLO11](https://docs.ultralytics.com/models/yolo11/), [licencias Ultralytics](https://www.ultralytics.com/license). La opción gratuita de Ultralytics usa AGPL-3.0; la licencia del proyecto y futura distribución siguen pendientes de decisión. No se activa una licencia empresarial.


## 1. Comprobar los recursos asignados

Esta celda no sube imágenes ni descarga pesos. Registra los recursos reales de esta sesión y prueba una operación en GPU. Si PyTorch no está disponible o CUDA falla, detenerse y conservar el error.


In [ ]:
import hashlib
import importlib.metadata
import json
from pathlib import Path
import platform
import subprocess
import sys
import time
import zipfile

import torch

print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print(subprocess.run(["free", "-h"], capture_output=True, text=True, check=True).stdout)
if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU CUDA disponible. Comprobar el entorno gratuito y detener el ensayo.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True, check=True).stdout)
probe = torch.ones((32, 32), device="cuda")
assert (probe @ probe)[0, 0].item() == 32
del probe
torch.cuda.empty_cache()
WORK = Path("/content/nox-pilot")
WORK.mkdir(exist_ok=True)
environment = {"python": sys.version, "platform": platform.platform(), "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)}
(WORK / "environment.json").write_text(json.dumps(environment, indent=2))
print("Comprobación GPU superada; todavía no hay entrenamiento.")


## 2. Instalar una versión fijada de Ultralytics

Se fija `ultralytics==8.4.189`, publicada al preparar el cuaderno. No se sustituye manualmente PyTorch de Colab. Se guardan todas las versiones resueltas y se comprueban dependencias; la compatibilidad real queda por verificar en esta sesión. Si se solicita reiniciar el entorno, reiniciarlo y volver a comenzar por la celda 1.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics==8.4.189"], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
from ultralytics import YOLO, settings
settings.update({"sync": False, "wandb": False, "mlflow": False, "comet": False, "clearml": False, "neptune": False})
environment["ultralytics"] = importlib.metadata.version("ultralytics")
(WORK / "environment.json").write_text(json.dumps(environment, indent=2))
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout
(WORK / "requirements-resolved.txt").write_text(freeze)
print("Ultralytics:", environment["ultralytics"])


## 3. Subir el paquete local y comprobarlo

**Ejecutar una vez registrados los permisos de uso/subida del material y revisada la licencia.** Elegir únicamente `nox-exploratory-dataset-2026-10-26.zip`, preparado en la carpeta local ignorada `data/review/colab-transfer/`. No subir vídeos completos ni publicar el ZIP en GitHub. La transferencia es manual a tu sesión de Colab.

Se rechazan archivos inesperados, alteraciones de imágenes/etiquetas y grupos compartidos entre particiones. Se reescribe la ruta Windows del YAML para Colab. Esto no confirma independencia visual de partidos ni suficiencia estadística.


In [ ]:
from google.colab import files
from PIL import Image
import yaml

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Seleccionar exactamente el ZIP preparado.")
archive_name, archive_bytes = next(iter(uploaded.items()))
if archive_name != "nox-exploratory-dataset-2026-10-26.zip":
    raise ValueError("Nombre de paquete inesperado.")
import io
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)) or sum(info.file_size for info in archive.infolist()) > 150_000_000:
        raise ValueError("ZIP duplicado o demasiado grande.")
    manifest_bytes = archive.read("manifest.json")
    expected_manifest = "3111e533e1a6be422c3fc30ed1c0e43a61b4748c7ba185416046f7c20108f571"
    assert hashlib.sha256(manifest_bytes).hexdigest() == expected_manifest, "Manifiesto inesperado"
    manifest = json.loads(manifest_bytes)
    expected_files = {"manifest.json", "verification.json", "dataset.yaml"}
    expected_files.update(item[key] for item in manifest["images"] for key in ("image", "label"))
    assert set(names) == expected_files, "Contenido de ZIP inesperado"
    DATA = WORK / "dataset"
    if DATA.exists():
        raise FileExistsError("Dataset ya extraído; no sobrescribir. Usar la celda siguiente si ya se verificó.")
    for info in archive.infolist():
        target = (DATA / info.filename).resolve()
        assert target.is_relative_to(DATA.resolve()), "Ruta no válida"
        if info.file_size > 15_000_000:
            raise ValueError("Archivo individual demasiado grande")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(info))

for item in manifest["images"]:
    image_path, label_path = DATA / item["image"], DATA / item["label"]
    assert hashlib.sha256(image_path.read_bytes()).hexdigest() == item["source_sha256"]
    assert hashlib.sha256(label_path.read_bytes()).hexdigest() == item["label_sha256"]
    with Image.open(image_path) as image:
        assert image.size == (item["width"], item["height"])
    rows = label_path.read_text().splitlines()
    assert len(rows) == len(item["boxes_xyxy"])
    for row in rows:
        label, *values = row.split()
        assert label == "0" and len(values) == 4
        cx, cy, width, height = map(float, values)
        assert 0 < width <= 1 and 0 < height <= 1
        assert 0 <= cx - width / 2 <= cx + width / 2 <= 1
        assert 0 <= cy - height / 2 <= cy + height / 2 <= 1
for key in ("source_sha256", "video_id", "group_id"):
    train_keys = {item[key] for item in manifest["images"] if item["split"] == "train"}
    val_keys = {item[key] for item in manifest["images"] if item["split"] == "val"}
    assert not train_keys.intersection(val_keys)
assert len(manifest["images"]) == 18
assert manifest["summary"]["train"]["images"] == 15
assert manifest["summary"]["val"]["images"] == 3
(DATA / "dataset.yaml").write_text(yaml.safe_dump({"path": str(DATA), "train": "images/train", "val": "images/val", "names": {0: "nox"}}))
print("18 imágenes y etiquetas verificadas:", manifest["summary"])


## 4. Comprobar la lectura y visualizar una etiqueta por partición

Se invoca el comprobador de configuración de Ultralytics y se muestra un positivo de cada partición. El cargador de entrenamiento completo se prueba al iniciar el bloque 6.


In [ ]:
from ultralytics.data.utils import check_det_dataset
from PIL import ImageDraw
from IPython.display import display

dataset_config = check_det_dataset(str(DATA / "dataset.yaml"), autodownload=False)
assert dataset_config["names"] == {0: "nox"}
for split in ("train", "val"):
    item = next(item for item in manifest["images"] if item["split"] == split and item["positive"])
    image = Image.open(DATA / item["image"]).convert("RGB")
    drawing = ImageDraw.Draw(image)
    for row in (DATA / item["label"]).read_text().splitlines():
        _, cx, cy, width, height = map(float, row.split())
        drawing.rectangle(((cx-width/2)*image.width, (cy-height/2)*image.height, (cx+width/2)*image.width, (cy+height/2)*image.height), outline="lime", width=4)
    print(split, item["video_id"], item["source_sha256"])
    display(image)


## 5. Elegir dónde guardar los checkpoints

Con `USE_DRIVE = True` se pedirá autorizar tu Drive y los checkpoints se escribirán allí durante el ensayo. Es opcional; la opción predeterminada usa el almacenamiento temporal de Colab. **Si la sesión se pierde sin Drive, se pueden perder los resultados aunque existan checkpoints locales.** El ZIP final solo protege lo que se descargue. No sobrescribir ejecuciones anteriores.


In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = Path("/content/drive/MyDrive/nox-pilot-runs")
else:
    RUNS = WORK / "runs"
RUN_NAME = "smoke-" + time.strftime("%Y%m%d-%H%M%S")
RUN = RUNS / RUN_NAME
if RUN.exists():
    raise FileExistsError(RUN)
RUN.mkdir(parents=True)
for filename in ("environment.json", "requirements-resolved.txt"):
    (RUN / filename).write_bytes((WORK / filename).read_bytes())
(RUN / "dataset-manifest.json").write_bytes((DATA / "manifest.json").read_bytes())
print("Resultados:", RUN)


## 6. Ejecutar un ensayo corto de entrenamiento

YOLO11n preentrenado; 3 épocas, resolución 640 y lote de 2. Son parámetros iniciales para probar el proceso, no una configuración optimizada. A esta resolución algunos logos pequeños pueden perder detalle. Se descargan pesos oficiales automáticamente; se registra su hash. Si falla por memoria, conservar el error antes de cambiar parámetros.

Se guarda un checkpoint por época. Esto no reanuda automáticamente una sesión interrumpida: para reanudar se necesitan el `last.pt` conservado, los datos y el entorno reconstruidos. No presentar estas métricas como evaluación final.


In [ ]:
model = YOLO("yolo11n.pt")
weights_path = Path(model.ckpt_path)
(RUN / "initial-weights-sha256.txt").write_text(hashlib.sha256(weights_path.read_bytes()).hexdigest() + "  yolo11n.pt\n")
started = time.monotonic()
results = model.train(data=str(DATA / "dataset.yaml"), epochs=3, imgsz=640, batch=2, device=0, workers=0, cache=False, project=str(RUNS), name=RUN_NAME, exist_ok=True, seed=42, deterministic=True, save=True, save_period=1, plots=True, amp=False)
report = {"elapsed_seconds": time.monotonic()-started, "epochs_requested": 3, "dataset_images": 18, "independent_test": False, "status": "exploratory_smoke_training_only"}
(RUN / "execution.json").write_text(json.dumps(report, indent=2))
assert (RUN / "weights/last.pt").is_file()
assert (RUN / "results.csv").is_file()
print(report)


## 7. Descargar y revisar los resultados

Descargar inmediatamente este ZIP y conservar una copia del cuaderno ejecutado. Puede contener pesos y vistas de imágenes: mantenerlo privado y fuera de Git hasta resolver su distribución. No hay métrica de test independiente. El siguiente bloque será revisar errores visuales y decidir cómo ampliar los datos, sin confundir este ensayo con la aceptación del nivel esencial.


In [ ]:
import shutil

result_archive = shutil.make_archive(str(WORK / (RUN_NAME + "-results")), "zip", RUN)
files.download(result_archive)
print("Descargar también el cuaderno ejecutado: Archivo → Descargar → Descargar .ipynb")
